In [1]:
"""
================================================================================
 sostenibilidad_defensa.py  —  Opción 2 (preparación de la defensa)
================================================================================
 🎯 QUÉ HACE
   Mide con codecarbon el COSTE ENERGÉTICO y las EMISIONES del entrenamiento del
   modelo ganador (LightGBM). Entrena un clon de usar y tirar (mismos
   hiperparámetros y random_state, leídos del propio .pkl) y guarda la medición.

 📋 REQUISITOS
   - Entorno conda: tfm_abandono (Python 3.11), con codecarbon, joblib, sklearn,
     lightgbm, pandas, pyarrow ya instalados.
   - Ejecutar EN ESPAÑA no es necesario gracias a OfflineEmissionsTracker:
     se fija el país a mano (ESP), así da igual la ciudad o si hay VPN.

 📤 GENERA
   data/06_evaluacion/sostenibilidad_defensa/emissions.csv  (medición real)
   Imprime por pantalla kWh y kg CO₂eq.

 ⚠️ SEGURIDAD (no toca nada congelado)
   - NO sobrescribe LightGBM__none.pkl ni metricas_modelo.json.
   - NO re-ejecuta la selección de modelos.
   - Solo entrena un clon en memoria y escribe el CSV en carpeta scratch.

 🔄 FLUJO
   1) localiza ROOT (sube buscando carpeta src/)
   2) carga el pipeline ganador desde el .pkl   -> de aquí saca los hiperparámetros
   3) carga X_train y y_train (RUTAS A CONFIRMAR abajo)
   4) clona el pipeline (resetea el estado entrenado, conserva params)
   5) envuelve el .fit() con OfflineEmissionsTracker(country_iso_code="ESP")
   6) imprime y guarda la medición
================================================================================
"""


from pathlib import Path
import joblib
import pandas as pd
from sklearn.base import clone
from codecarbon import OfflineEmissionsTracker

# ----------------------------------------------------------------------------
# 1) Localizar la raíz del repo (sube buscando la carpeta src/)
# ----------------------------------------------------------------------------

ROOT = Path.cwd()
for _ in range(8):
    if (ROOT / "src").is_dir():
        break
    ROOT = ROOT.parent
print(f"ROOT detectada: {ROOT}")

# ============================================================================
# ⚠️  AJUSTA / CONFIRMA SOLO ESTAS RUTAS  (lo demás no se toca)
# ============================================================================
MODEL_PATH = ROOT / "data" / "05_modelado" / "models" / "LightGBM__none.pkl"

# Rutas CONFIRMADas contra el contenido real de data/05_modelado:
#   - X_train_prep.parquet existe (features de entrenamiento ya preparadas)
#   - y_train.parquet existe (target en fichero aparte; NO va dentro de X)
X_TRAIN_PATH = ROOT / "data" / "05_modelado" / "X_train_prep.parquet"
Y_TRAIN_PATH = ROOT / "data" / "05_modelado" / "y_train.parquet"
TARGET_COL = None   # el target va en fichero aparte, no como columna de X
# ============================================================================

OUTPUT_DIR = ROOT / "data" / "06_evaluacion" / "sostenibilidad_defensa"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------------
# 2) Cargar el pipeline ganador (de aquí salen los hiperparámetros)
# ----------------------------------------------------------------------------
pipeline = joblib.load(MODEL_PATH)
print("Pipeline ganador cargado:", type(pipeline).__name__)

# ----------------------------------------------------------------------------
# 3) Cargar datos de entrenamiento
# ----------------------------------------------------------------------------
X_train = pd.read_parquet(X_TRAIN_PATH)
y_train = pd.read_parquet(Y_TRAIN_PATH)
# si y viene como DataFrame de una sola columna, lo aplanamos a Series
if hasattr(y_train, "columns"):
    y_train = y_train.iloc[:, 0]

print(f"X_train: {X_train.shape}  |  y_train: {y_train.shape}")

# ----------------------------------------------------------------------------
# 4) Clonar el pipeline (mismos params, sin estado entrenado)
# ----------------------------------------------------------------------------
modelo = clone(pipeline)

# ----------------------------------------------------------------------------
# 5) Medir el entrenamiento con codecarbon (país fijado a España)
# ----------------------------------------------------------------------------
tracker = OfflineEmissionsTracker(
    country_iso_code="ESP",              # <- fija España: ni VPN ni IP cambian esto
    project_name="TFM_defensa_LightGBM",
    output_dir=str(OUTPUT_DIR),
    output_file="emissions.csv",
    measure_power_secs=1,
    log_level="error",
)

tracker.start()
modelo.fit(X_train, y_train)
emisiones_kg = tracker.stop()

# ----------------------------------------------------------------------------
# 6) Resultado
# ----------------------------------------------------------------------------
print("\n" + "=" * 60)
print(" RESULTADO codecarbon — entrenamiento LightGBM (España)")
print("=" * 60)
print(f" Emisiones : {emisiones_kg} kg CO2eq")
print(f" CSV       : {OUTPUT_DIR / 'emissions.csv'}")
print(" (energía en kWh y detalles completos: ver columna energy_consumed del CSV)")
print("=" * 60)


ROOT detectada: c:\FF\AU_UJI_v2


[codecarbon INFO @ 20:52:08] offline tracker init
[codecarbon WARNING @ 20:52:08] Multiple instances of codecarbon are allowed to run at the same time.


Pipeline ganador cargado: Pipeline
X_train: (26896, 27)  |  y_train: (26896,)

 RESULTADO codecarbon — entrenamiento LightGBM (España)
 Emisiones : 4.309704474250438e-06 kg CO2eq
 CSV       : c:\FF\AU_UJI_v2\data\06_evaluacion\sostenibilidad_defensa\emissions.csv
 (energía en kWh y detalles completos: ver columna energy_consumed del CSV)
